
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Task 3 - Investigate and Resolve Unusual Patterns
In this task, we investigate unusual patterns detected in the dataset, such as high cardinality in certain columns or skewed distributions. These patterns can affect model performance and should be analyzed to determine if corrective action is needed. This notebook summarizes the identified unusual patterns, offers suggestions for handling them, and applies transformations to resolve them.

**Objectives:**

- Review columns with high cardinality and skewed distributions.
- Apply transformations to address these unusual patterns.
- Document and suggest steps to mitigate issues caused by unusual patterns.
- Perform feature importance analysis to identify key features affecting the target variable.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **17.3.x-cpu-ml-scala2.13**

## Classroom Setup

Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo. Execute the following cell:

In [0]:
%pip install category_encoders

In [0]:
%run ../../Includes/Classroom-Setup-1.1demo

**Other Conventions:**

Throughout this lab, we'll refer to the object DA. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"User DB Location:  {DA.paths.datasets}")


##Task Outline
In this task, we will:

- Load the unusual patterns report generated by the Alert_Unusual_Patterns task.
- Investigate high cardinality columns and assess their impact.
- Analyze skewed distributions and determine potential transformations.
- Document findings and suggest next steps.


###Step 1: Load the Unusual Patterns Report
We start by loading the unusual patterns report that was generated in the previous task. This report contains details about columns with high cardinality and skewed distributions.

In [0]:
# Define the path to the unusual patterns report
unusual_patterns_report_path = "./unusual_patterns_report.txt"

# Load and display the unusual patterns report
print("Loading Unusual Patterns Report...")
with open(unusual_patterns_report_path, "r") as f:
    unusual_patterns_report = f.read()

print("Unusual Patterns Report Loaded")
print("===================================")
print(unusual_patterns_report)


###Step 2: Parse and Investigate High Cardinality Columns
High cardinality in categorical features can increase model complexity and may lead to overfitting. We review the columns identified as having high cardinality and discuss potential techniques for managing them.

In [0]:
# Parse high cardinality columns from the report
high_cardinality_columns = []
for line in unusual_patterns_report.splitlines():
    if "High Cardinality Columns:" in line:
        high_cardinality_columns = line.split(":")[1].strip("[]").replace("'", "").split(", ")
        break

print("High Cardinality Columns Identified:")
print(high_cardinality_columns)

# Suggested actions for high cardinality columns
print("\nSuggested Actions for High Cardinality Columns:")
for col in high_cardinality_columns:
    print(f"- Column '{col}': Consider using frequency encoding, target encoding, or clustering to reduce cardinality.")

###Step 3: Parse and Analyze Skewed Distributions
Skewed distributions can negatively impact model performance, particularly for algorithms sensitive to outliers. We review the columns with skewed distributions and discuss transformations that can help normalize these distributions.

In [0]:
# Parse skewed columns from the report
skewed_columns = []
for line in unusual_patterns_report.splitlines():
    if "Skewed Columns:" in line:
        skewed_columns = line.split(":")[1].strip("[]").replace("'", "").split(", ")
        break

print("Skewed Columns Identified:")
print(skewed_columns)

# Suggested actions for skewed columns
print("\nSuggested Actions for Skewed Columns:")
for col in skewed_columns:
    if col:  # Only suggest actions if columns exist
        print(f"- Column '{col}': Consider applying transformations like log, square root, or Box-Cox to reduce skewness.")


###Step 4: Apply Transformations to Resolve Unusual Patterns
In this step, we apply target encoding to high cardinality columns and log transformations to skewed columns. This process helps to improve model performance by addressing issues caused by these unusual patterns.

In [0]:
# Define the dataset path
dataset_path = f"{DA.paths.datasets.banking}/banking/loan-clean.csv"

# Load the loan dataset
df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(dataset_path)

from pyspark.sql.functions import col, log1p
from category_encoders import TargetEncoder

# High Cardinality Columns
high_cardinality_columns = ['ID', 'Income', 'ZIP Code', 'CCAvg', 'Mortgage']

# Target column
target_column = 'Personal Loan'

# Apply Target Encoding for high cardinality columns
target_encoder = TargetEncoder(cols=high_cardinality_columns)
df_encoded = target_encoder.fit_transform(df.toPandas(), df.toPandas()[target_column])
df_encoded = spark.createDataFrame(df_encoded)

# Skewed Distribution Columns
skewed_columns = ['ZIP Code', 'CCAvg', 'Mortgage', 'Personal Loan', 'Securities Account', 'CD Account']

# Apply transformations to reduce skewness
df_transformed = df_encoded
for col_name in skewed_columns:
    df_transformed = df_transformed.withColumn(col_name, log1p(col(col_name)))

# Display the transformed DataFrame
display(df_transformed)

print("Unusual Patterns Resolved")


###Step 5: Document Findings and Next Steps
In this step, we save a summary of our findings and recommendations for each unusual pattern detected. This summary report provides guidance for preprocessing steps in the data pipeline.

In [0]:
# Define path for the final investigation report
investigation_report_path = "./investigate_unusual_patterns_report.txt"

print("Compiling Investigation Report...")

with open(investigation_report_path, "w") as f:
    f.write("Investigation Report on Unusual Patterns\n")
    f.write("========================================\n\n")
    
    # Document high cardinality columns
    f.write("High Cardinality Columns\n")
    f.write("------------------------\n")
    if high_cardinality_columns:
        f.write("The following columns have high cardinality:\n")
        for col in high_cardinality_columns:
            f.write(f"- Column '{col}': Suggested action: reduce cardinality using frequency encoding or target encoding.\n")
    else:
        f.write("No high cardinality columns detected.\n")
    f.write("\n")
    
    # Document skewed distribution columns
    f.write("Skewed Distribution Columns\n")
    f.write("---------------------------\n")
    if skewed_columns:
        f.write("The following columns have skewed distributions:\n")
        for col in skewed_columns:
            f.write(f"- Column '{col}': Suggested action: apply transformations like log, square root, or Box-Cox to reduce skewness.\n")
    else:
        f.write("No skewed distribution columns detected.\n")

print("Investigation report saved to:", investigation_report_path)


###Step 6: Display Report Summary
Display a summary of the investigation findings within the notebook for easy reference.

In [0]:
print("=== Investigation Report Summary ===")
print("\nHigh Cardinality Columns:")
print(high_cardinality_columns if high_cardinality_columns else "None detected")

print("\nSkewed Distribution Columns:")
print(skewed_columns if skewed_columns else "None detected")


###Step 7: Feature Importance Analysis
Finally, we perform a feature importance analysis to identify the most influential features for predicting the target variable. This step helps prioritize which features to focus on during further modeling and data preprocessing.

In [0]:
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer
from pyspark.sql.types import StringType
from sklearn.model_selection import train_test_split

# Define target and features
target = "Personal Loan"
X = df.drop("ID", "ZIP Code", target)  # Drop target and unnecessary columns
y = df.select(col(target).cast("int")).toPandas()[target]  # Convert target to integer and collect as pandas Series

# Encode categorical variables
categorical_cols = [field.name for field in X.schema.fields if isinstance(field.dataType, StringType)]

for col_name in categorical_cols:
    indexer = StringIndexer(inputCol=col_name, outputCol=col_name + "_indexed")
    X = indexer.fit(X).transform(X).drop(col_name)

# Convert X to pandas DataFrame for train_test_split
X = X.toPandas()

# Split the dataset for training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Initialize and train the model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Get feature importances
feature_importances = pd.Series(model.feature_importances_, index=X.columns)
important_features = feature_importances.nlargest(5)

print("Top 5 Important Features")
print("========================")
print(important_features)
import json

# Convert important features to DataFrame
important_features_df = important_features.reset_index()
important_features_df.columns = ["Feature", "Importance"]

# Convert the sample display data to JSON format for API access and inspection
output_data = important_features_df.head(5).to_dict(orient="records")

# Print the JSON-formatted output for the final task
print("Notebook_Output:", json.dumps(output_data, indent=4))

# Save the visualization data and sample output to a JSON file
output_json_path = "./feature_engineered_output_with_visualization_data.json"
with open(output_json_path, "w") as json_file:
    json.dump({"sample_data": output_data}, json_file, indent=4)

print(f"JSON output saved to: {output_json_path}")


##Conclusion
In this notebook, we:

- Investigated high cardinality columns, identifying potential techniques to reduce their impact.
- Reviewed columns with skewed distributions, suggesting transformations to improve model performance.
- Compiled findings into a detailed report for future reference.

This investigation provides a foundation for refining data preprocessing steps, ensuring that issues related to unusual patterns are addressed before proceeding with model training. The insights gained here can be used to guide feature engineering and data transformation in the MLOps pipeline.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>